# Hafta 9: Kümeleme Algoritmaları

Bu notebook'ta aşağıdaki konular ele alınmaktadır:
1. Veri seti keşfi ve ön işleme
2. K-Means kümeleme
3. Elbow Method ve Silhouette ile K seçimi
4. Hiyerarşik kümeleme (Agglomerative)
5. Dendrogram analizi
6. DBSCAN ile yoğunluk tabanlı kümeleme
7. İç metriklerle model değerlendirme
8. Algoritma karşılaştırması ve yorum

## 1. Gerekli Kütüphaneler

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

# scipy hiyerarşik çizim için
try:
    from scipy.cluster.hierarchy import dendrogram, linkage
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

print('Kütüphaneler yüklendi.')
print('scipy durumu:', 'Hazır' if HAS_SCIPY else 'Eksik (pip install scipy)')

## 2. Veri Seti Yükleme

Bu uygulamada `mall_customers.csv` kullanılıyor.

Sütunlar:
- `CustomerID`: müşteri kimliği
- `Gender`: cinsiyet
- `Age`: yaş
- `Annual_Income`: yıllık gelir
- `Spending_Score`: harcama skoru

In [ ]:
DATA_PATH = os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    '..', '..', 'Veri_Setleri_Genel', 'mall_customers.csv'
)

df = pd.read_csv(DATA_PATH)
print(f'Veri seti boyutu: {df.shape[0]} satır, {df.shape[1]} sütun')
display(df.head())

In [ ]:
print('Veri tipleri:')
print(df.dtypes)
print('\nEksik değer sayıları:')
print(df.isnull().sum())

print('\nTemel istatistikler:')
display(df.describe(include='all'))

In [ ]:
# Hızlı EDA
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df['Age'], bins=20, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Yaş Dağılımı')

sns.histplot(df['Annual_Income'], bins=20, kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('Yıllık Gelir Dağılımı')

sns.histplot(df['Spending_Score'], bins=20, kde=True, ax=axes[2], color='tomato')
axes[2].set_title('Harcama Skoru Dağılımı')

plt.tight_layout()
plt.show()

## 3. Ön İşleme

Kümeleme algoritmaları mesafeye dayalı olduğu için ölçekleme kritik önem taşır.

In [ ]:
# Sayısal öznitelikler
feature_cols = ['Age', 'Annual_Income', 'Spending_Score']
X = df[feature_cols].copy()

# Standartlaştırma
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Kullanılan öznitelikler:', feature_cols)
print('Ölçeklenmiş veri boyutu:', X_scaled.shape)

## 4. K-Means için K Seçimi

### Elbow Method
Inertia (WCSS) değerini farklı K değerleri için hesaplayıp dirsek noktasını ararız.

### Silhouette Score
$$s = \frac{b-a}{\max(a,b)}$$
- $s n [-1,1]$
- Yüksek değer daha iyi ayrışmayı gösterir

In [ ]:
k_values = range(2, 11)
inertias = []
sil_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(k_values, inertias, 'o-', color='darkorange')
axes[0].set_title('Elbow Method (Inertia)')
axes[0].set_xlabel('K (Küme sayısı)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_values, sil_scores, 's-', color='teal')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('K (Küme sayısı)')
axes[1].set_ylabel('Silhouette')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_k = k_values[int(np.argmax(sil_scores))]
print(f'Silhouette skoruna göre en iyi K: {best_k}')

## 5. K-Means Kümeleme

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
labels_kmeans = kmeans.fit_predict(X_scaled)

# Değerlendirme
sil_km = silhouette_score(X_scaled, labels_kmeans)
dbi_km = davies_bouldin_score(X_scaled, labels_kmeans)
chi_km = calinski_harabasz_score(X_scaled, labels_kmeans)

print('K-Means metrikleri:')
print(f'  Silhouette           : {sil_km:.4f}')
print(f'  Davies-Bouldin       : {dbi_km:.4f}')
print(f'  Calinski-Harabasz    : {chi_km:.4f}')

df_km = df.copy()
df_km['Cluster_KMeans'] = labels_kmeans
display(df_km.groupby('Cluster_KMeans')[feature_cols].mean().round(2))

In [ ]:
# 2D görselleştirme (PCA)
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

centers_2d = pca.transform(kmeans.cluster_centers_)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_kmeans, cmap='viridis', s=60, alpha=0.85)
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], marker='X', s=250, c='red', label='Merkezler')
plt.title(f'K-Means Sonucu (K={best_k}) - PCA 2D')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

## 6. Hiyerarşik Kümeleme (Agglomerative)

Aynı K değeri ile Ward linkage kullanarak kümeleme yapılır.

In [ ]:
agg = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
labels_agg = agg.fit_predict(X_scaled)

sil_agg = silhouette_score(X_scaled, labels_agg)
dbi_agg = davies_bouldin_score(X_scaled, labels_agg)
chi_agg = calinski_harabasz_score(X_scaled, labels_agg)

print('Agglomerative metrikleri:')
print(f'  Silhouette           : {sil_agg:.4f}')
print(f'  Davies-Bouldin       : {dbi_agg:.4f}')
print(f'  Calinski-Harabasz    : {chi_agg:.4f}')

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_agg, cmap='plasma', s=60, alpha=0.85)
plt.title(f'Agglomerative Clustering (K={best_k}) - PCA 2D')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
# Dendrogram
if HAS_SCIPY:
    Z = linkage(X_scaled, method='ward')
    plt.figure(figsize=(12, 5))
    dendrogram(Z, truncate_mode='lastp', p=15, leaf_rotation=45, leaf_font_size=10, show_contracted=True)
    plt.title('Hiyerarşik Kümeleme Dendrogram (Ward)')
    plt.xlabel('Örnek / Küme')
    plt.ylabel('Mesafe')
    plt.tight_layout()
    plt.show()
else:
    print('Dendrogram için scipy gerekli: pip install scipy')

## 7. DBSCAN (Yoğunluk Tabanlı Kümeleme)

DBSCAN için temel parametreler:
- `eps`: komşuluk yarıçapı
- `min_samples`: çekirdek nokta için minimum komşu sayısı

In [ ]:
# eps seçimine yardımcı k-distance grafiği
neighbors = NearestNeighbors(n_neighbors=5)
nbrs = neighbors.fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)

kdist = np.sort(distances[:, 4])
plt.figure(figsize=(8, 4))
plt.plot(kdist, color='slateblue')
plt.title('DBSCAN için 5-NN Mesafe Grafiği')
plt.xlabel('Nokta sıralaması')
plt.ylabel('5. komşu mesafesi')
plt.grid(True, alpha=0.3)
plt.show()

print('Grafikte kırılma noktasına yakın bir eps seçebilirsiniz (ör. 0.8-1.2 arası).')

In [ ]:
dbscan = DBSCAN(eps=1.0, min_samples=5)
labels_db = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = np.sum(labels_db == -1)

print(f'DBSCAN küme sayısı (noise hariç): {n_clusters_db}')
print(f'Noise nokta sayısı: {n_noise}')

if n_clusters_db >= 2:
    sil_db = silhouette_score(X_scaled, labels_db)
    dbi_db = davies_bouldin_score(X_scaled, labels_db)
    chi_db = calinski_harabasz_score(X_scaled, labels_db)
    print(f'  Silhouette           : {sil_db:.4f}')
    print(f'  Davies-Bouldin       : {dbi_db:.4f}')
    print(f'  Calinski-Harabasz    : {chi_db:.4f}')
else:
    sil_db, dbi_db, chi_db = np.nan, np.nan, np.nan
    print('DBSCAN için yeterli küme oluşmadı; eps/min_samples ayarını değiştirin.')

In [ ]:
plt.figure(figsize=(8, 6))
unique_labels = np.unique(labels_db)
palette = sns.color_palette('tab10', n_colors=max(len(unique_labels), 3))

for idx, label in enumerate(unique_labels):
    mask = labels_db == label
    if label == -1:
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1], c='black', s=40, label='Noise', alpha=0.7)
    else:
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[palette[idx]], s=60, label=f'Küme {label}', alpha=0.85)

plt.title('DBSCAN Sonucu - PCA 2D')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

## 8. Algoritma Karşılaştırması

İç metrikler:
- Silhouette: yüksek iyi
- Davies-Bouldin: düşük iyi
- Calinski-Harabasz: yüksek iyi

In [ ]:
results = pd.DataFrame({
    'Algoritma': ['K-Means', 'Agglomerative', 'DBSCAN'],
    'Silhouette': [sil_km, sil_agg, sil_db],
    'Davies-Bouldin': [dbi_km, dbi_agg, dbi_db],
    'Calinski-Harabasz': [chi_km, chi_agg, chi_db]
})

display(results.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(data=results, x='Algoritma', y='Silhouette', ax=axes[0], palette='Blues_d')
axes[0].set_title('Silhouette (yüksek iyi)')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=results, x='Algoritma', y='Davies-Bouldin', ax=axes[1], palette='Reds_d')
axes[1].set_title('Davies-Bouldin (düşük iyi)')
axes[1].tick_params(axis='x', rotation=20)

sns.barplot(data=results, x='Algoritma', y='Calinski-Harabasz', ax=axes[2], palette='Greens_d')
axes[2].set_title('Calinski-Harabasz (yüksek iyi)')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 9. Sonuç

Bu uygulamada üç kümeleme algoritması karşılaştırılmıştır:
- **K-Means**: hızlı, basit ve büyük veride etkili
- **Hiyerarşik Kümeleme**: dendrogram ile yorumlanabilir
- **DBSCAN**: gürültü/noise noktalarını tespit eder

Pratik seçim önerisi:
1. Veri küresel kümeler içeriyorsa K-Means ile başlayın.
2. Küme sayısı konusunda emin değilseniz hiyerarşik yöntem + dendrogram kullanın.
3. Outlier/noise kritikse DBSCAN deneyin.
4. Son kararı tek metrikle değil, birden çok metrik + iş bilgisi ile verin.